# Santander Case

In [161]:
#Libraries

#Data handling and cleaning
import pandas as pd
import numpy as np


#Import data from local path
from pathlib import Path

#Visualizations

import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.figure_factory as ff

#VIF
from statsmodels.stats.outliers_influence import variance_inflation_factor 
from statsmodels.tools.tools import add_constant 


#ML Models Libraries
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix, roc_curve
from sklearn.model_selection import GridSearchCV
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA


#Artificial data (SMOTE)
from imblearn.over_sampling import SMOTE

Import Raw DataFrame


In [162]:
data_path = Path("data") / "NPS-Mexico-2026-v1.xlsx"

if not data_path.exists():
    raise FileNotFoundError(
        f"Dataset not found at {data_path}. Add your file to data/NPS-Mexico-2026-v1.xlsx"
    )

df_raw = pd.read_excel(data_path)

## **<u> Exploratory Dataset Analysis </u>**

In [163]:
#Dataset columns obtained from the survey
df_raw.columns

Index(['Marca temporal',
       '¿Qué institución financiera desea evaluar en esta encuesta (su institución principal o de experiencia reciente)?',
       'En los últimos 3 meses, ¿ha realizado al menos una operación o interacción con esta institución (app/web/sucursal/cajero/teléfono)?',
       '¿Cuánto tiempo lleva siendo cliente de esta institución?',
       '¿Cuáles de los siguientes productos/servicios ha utilizado con esta institución en los últimos 3 meses?',
       '¿A través de qué canales ha realizado la mayoría de sus operaciones con esta institución en los últimos 3 meses?',
       '¿Cuál es su canal principal para interactuar con esta institución?',
       'Pensando en su experiencia reciente, ¿qué tan de acuerdo está con la frase: “Resolver lo que necesitaba con esta institución fue fácil”?',
       'En su interacción más reciente con esta institución, ¿su necesidad se resolvió en el primer contacto (sin tener que insistir o repetir el proceso)?',
       'En una escala de

In [164]:
#Main statistical metrics per numerical column
df_raw.describe()

,Marca temporal,"Pensando en su experiencia reciente, ¿qué tan de acuerdo está con la frase: “Resolver lo que necesitaba con esta institución fue fácil”?","En general, ¿qué tan satisfecho(a) está con esta institución?","En una escala de 0 a 10, ¿qué tan probable es que recomiende esta institución a un familiar o amigo?"
count,193,193.000000,193.000000,193.000000
mean,2026-04-03 07:09:00.480927488,3.901554,4.134715,8.191710
min,2026-03-11 16:55:42.769000,1.000000,1.000000,0.000000
25%,2026-03-24 20:08:28.419000064,3.000000,4.000000,8.000000
50%,2026-04-08 14:03:16.892999936,4.000000,4.000000,9.000000
75%,2026-04-08 18:13:01.548000,5.000000,5.000000,10.000000
max,2026-04-15 00:31:48.138000,5.000000,5.000000,10.000000
std,NaN,1.210048,0.925612,2.086548


In [165]:
#Data types

data_types = pd.DataFrame(df_raw.dtypes)

data_types

,0
Marca temporal,datetime64[ns]
¿Qué institución financiera desea evaluar en esta encuesta (su institución principal o de experiencia reciente)?,object
"En los últimos 3 meses, ¿ha realizado al menos una operación o interacción con esta institución (app/web/sucursal/cajero/teléfono)?",object
¿Cuánto tiempo lleva siendo cliente de esta institución?,object
¿Cuáles de los siguientes productos/servicios ha utilizado con esta institución en los últimos 3 meses?,object
¿A través de qué canales ha realizado la mayoría de sus operaciones con esta institución en los últimos 3 meses?,object
¿Cuál es su canal principal para interactuar con esta institución?,object
"Pensando en su experiencia reciente, ¿qué tan de acuerdo está con la frase: “Resolver lo que necesitaba con esta institución fue fácil”?",int64
"En su interacción más reciente con esta institución, ¿su necesidad se resolvió en el primer contacto (sin tener que insistir o repetir el proceso)?",object
"En una escala del 1 al 5, califique su experiencia con esta institución en cada aspecto: [Rapidez de atención / respuesta]",object


## **<u> Data Standardization </u>**

In [166]:
from data_processing import data_processing

df_est = data_processing()

In [167]:
df_est.columns

Index(['marca_temporal', 'institucion', 'operacion_reciente', 'antiguedad',
       'productos_usados', 'canales_usados', 'canal_principal', 'ces',
       'resolucion_primer_contacto', 'cal_rapidez', 'cal_transparencia',
       'cal_facilidad_digital', 'cal_estabilidad_digital', 'cal_confianza',
       'cal_atencion_humana', 'csat', 'nps', 'razon_nps', 'edad', 'estado',
       'nivel_estudios', 'ocupacion', 'productos_usados_lista',
       'canales_usados_lista', 'razon_nps_lista', 'nps_categoria', 'mes',
       'trimestre'],
      dtype='object')

In [168]:
df_est.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 193 entries, 0 to 192
Data columns (total 28 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   marca_temporal              193 non-null    datetime64[ns]
 1   institucion                 193 non-null    category      
 2   operacion_reciente          193 non-null    category      
 3   antiguedad                  193 non-null    category      
 4   productos_usados            193 non-null    object        
 5   canales_usados              193 non-null    object        
 6   canal_principal             193 non-null    category      
 7   ces                         193 non-null    int64         
 8   resolucion_primer_contacto  193 non-null    category      
 9   cal_rapidez                 193 non-null    category      
 10  cal_transparencia           193 non-null    category      
 11  cal_facilidad_digital       193 non-null    category      

## **<u> Exploratory Analysis (Standardized DF) </u>**

In [169]:
#List of DF columns
print(df_est.columns.tolist())

['marca_temporal', 'institucion', 'operacion_reciente', 'antiguedad', 'productos_usados', 'canales_usados', 'canal_principal', 'ces', 'resolucion_primer_contacto', 'cal_rapidez', 'cal_transparencia', 'cal_facilidad_digital', 'cal_estabilidad_digital', 'cal_confianza', 'cal_atencion_humana', 'csat', 'nps', 'razon_nps', 'edad', 'estado', 'nivel_estudios', 'ocupacion', 'productos_usados_lista', 'canales_usados_lista', 'razon_nps_lista', 'nps_categoria', 'mes', 'trimestre']


In [170]:
#Confirm key data types
print(df_est[["ces", "csat", "nps"]].dtypes)

ces     int64
csat    int64
nps     int64
dtype: object


In [171]:
#Quick overview of main metrics
print(df_est[["ces", "csat", "nps"]].describe())

              ces        csat         nps
count  193.000000  193.000000  193.000000
mean     3.901554    4.134715    8.191710
std      1.210048    0.925612    2.086548
min      1.000000    1.000000    0.000000
25%      3.000000    4.000000    8.000000
50%      4.000000    4.000000    9.000000
75%      5.000000    5.000000   10.000000
max      5.000000    5.000000   10.000000


## **<u> Hypothesis Development Based on Main Focus </u>**

### NPS Drivers and Passive-to-Promoter Conversion

Proposed Hypothesis

 Trust, transparency, and friction (CES) variables explain
 the difference between a Passive and a Promoter customer.

Data Preparation

In [172]:
#Ordinal encoding dictionary — maps each satisfaction level
#to a number from 1 to 5, worst to best.
#"no aplica" maps to None so it becomes NaN and doesn't affect the analysis.
orden_cal = {
    "muy mala": 1, "mala": 2, "regular": 3,
    "buena": 4, "muy buena": 5, "no aplica": None
}

#Copy of the standardized df to avoid modifying the original.
#All NPS model work is done on df_nps.
df_nps = df_est.copy()


#Encoding of ordinal perceived-quality variables.
#New columns are created with a _num suffix to keep
#the original column intact and have the numeric version
#available for correlations and models.
df_nps["cal_confianza_num"]           = df_nps["cal_confianza"].map(orden_cal).astype(float)
df_nps["cal_transparencia_num"]       = df_nps["cal_transparencia"].map(orden_cal).astype(float)
df_nps["cal_rapidez_num"]             = df_nps["cal_rapidez"].map(orden_cal).astype(float)
df_nps["cal_facilidad_digital_num"]   = df_nps["cal_facilidad_digital"].map(orden_cal).astype(float)
df_nps["cal_estabilidad_digital_num"] = df_nps["cal_estabilidad_digital"].map(orden_cal).astype(float)
df_nps["cal_atencion_humana_num"]     = df_nps["cal_atencion_humana"].map(orden_cal).astype(float)


#Filter df to keep only Passives and Promoters.
#Detractors are excluded because the hypothesis is specifically
#about what separates these two groups.
df_pasivos_promotores = df_nps[
    df_nps["nps_categoria"].isin(["Pasivo (7-8)", "Promotor (9-10)"])
].copy()

#When filtering a Categorical, unused categories still exist
#in the index. remove_unused_categories() drops them so they don't
#show up empty in plots and groupings.
df_pasivos_promotores["nps_categoria"] = df_pasivos_promotores["nps_categoria"].cat.remove_unused_categories()

#Binary target variable for correlation:
#Passive = 0, Promoter = 1.
#This lets us measure which variables push toward being a Promoter.
df_pasivos_promotores["es_promotor"] = (
    df_pasivos_promotores["nps_categoria"] == "Promotor (9-10)"
).astype(int)

Plot 1 - Distribution of Detractors, Passives, and Promoters

In [173]:
orden  = ["Detractor (0-6)", "Pasivo (7-8)", "Promotor (9-10)"]
label_map = {
    "Detractor (0-6)": "Detractor (0-6)",
    "Pasivo (7-8)": "Passive (7-8)",
    "Promotor (9-10)": "Promoter (9-10)"
}

counts = df_nps["nps_categoria"].value_counts().reindex(orden).fillna(0).astype(int).reset_index()
counts.columns = ["cat", "n"]
counts["cat_label"] = counts["cat"].map(label_map)
counts["label"] = counts.apply(lambda r: f"{r.n}<br>({r.n/len(df_nps)*100:.1f}%)", axis=1)

orden_en = ["Detractor (0-6)", "Passive (7-8)", "Promoter (9-10)"]

fig1 = px.bar(counts, x="cat_label", y="n", color="cat_label", text="label",
              color_discrete_sequence=["#e63946", "#f4a261", "#2a9d8f"],
              category_orders={"cat_label": orden_en},
              title="Customer distribution by NPS category",
              labels={"n": "Number of customers", "cat_label": ""})

fig1.update_traces(textposition="outside", cliponaxis=False, width=0.55)
fig1.update_layout(
    plot_bgcolor="#F5F5F0",
    paper_bgcolor="#F5F5F0",
    font=dict(color="#333333", size=12),
    title_font=dict(size=16, color="#1a1a2e", family="Arial Black"),
    xaxis=dict(tickfont=dict(color="#333333", size=12), linecolor="#cccccc"),
    yaxis=dict(gridcolor="#e0e0e0", tickfont=dict(color="#333333", size=12), zeroline=False),
    showlegend=False
)

fig1.show()


Plot 2 - Variable Correlation with NPS

In [174]:
hipotesis = ["cal_confianza_num", "cal_transparencia_num"]

df_corr = (df_pasivos_promotores.select_dtypes("number")
           .corr(method="spearman")["es_promotor"]
           .drop(["es_promotor", "nps"], errors="ignore")
           .sort_values()
           .rename_axis("var").reset_index(name="corr"))

df_corr["color"] = df_corr["corr"].apply(lambda x: "#e63946" if x < 0 else "#2a9d8f")
df_corr["tick"]  = df_corr["var"].apply(
    lambda v: f"<b><span style='color:#f4a261'>{v}</span></b>" if v in hipotesis
              else f"<span style='color:#333333'>{v}</span>")

fig2 = px.bar(df_corr, x="corr", y="var", orientation="h", color="var",
              color_discrete_map=dict(zip(df_corr["var"], df_corr["color"])),
              text=df_corr["corr"].apply(lambda x: f"{x:.2f}"),
              category_orders={"var": df_corr["var"].tolist()},
              title="What separates a Passive from a Promoter?",
              labels={"corr": "Correlation with being a Promoter (Spearman)", "var": ""})

fig2.update_traces(textposition="outside", cliponaxis=False)
fig2.update_layout(
    plot_bgcolor="#F5F5F0",
    paper_bgcolor="#F5F5F0",
    font=dict(color="#333333", size=12),
    title_font=dict(size=16, color="#1a1a2e", family="Arial Black"),
    xaxis=dict(tickfont=dict(color="#333333", size=12), linecolor="#cccccc",
               title="Correlation with being a Promoter (Spearman)"),
    yaxis=dict(ticktext=df_corr["tick"].tolist(), tickvals=df_corr["var"].tolist(),
               tickfont=dict(color="#333333"), gridcolor="#e0e0e0", zeroline=False),
    showlegend=False,
    shapes=[dict(type="line", x0=0, x1=0, y0=-0.5, y1=len(df_corr)-0.5,
                 line=dict(color="#333333", width=0.8, dash="dash"), opacity=0.5)]
)

fig2.show()

Plot 3 - Comparison between Passives and Promoters by variable

In [175]:
#Comparative profile — Passives vs Promoters across all numerical variables
#nps, es_promotor, mes, trimestre, ces, csat are excluded
variables_perfil = [col for col in df_pasivos_promotores.select_dtypes("number").columns
                    if col not in ["nps", "es_promotor", "mes", "trimestre"]]

#Average of each variable by segment
perfil = (
    df_pasivos_promotores.groupby("nps_categoria")[variables_perfil]
    .mean()
    .round(2)
    .T
    .reset_index()
    .rename(columns={"index": "variable"})
)

#Long format for plotly
perfil_melted = perfil.melt(
    id_vars="variable",
    var_name="segmento",
    value_name="promedio"
)


fig = px.bar(
    perfil_melted,
    x="variable",
    y="promedio",
    color="segmento",
    barmode="group",
    text="promedio",
    title="Comparative profile — Passives vs Promoters",
    labels={"variable": "", "promedio": "Average score (1-5)", "segmento": ""},
    color_discrete_map={
        "Pasivo (7-8)":    "#f4a261",
        "Promotor (9-10)": "#2a9d8f"
    }
)

fig.update_traces(textposition="outside", textfont=dict(size=10))
fig.update_layout(
    plot_bgcolor="#F5F5F0",
    paper_bgcolor="#F5F5F0",
    font=dict(color="#333333", size=12),
    title_font=dict(size=16, color="#1a1a2e", family="Arial Black"),
    xaxis=dict(tickfont=dict(color="#333333", size=11), linecolor="#cccccc"),
    yaxis=dict(gridcolor="#e0e0e0", tickfont=dict(color="#333333", size=11),
               range=[0, 6]),
    legend=dict(font=dict(color="#333333")),
    width=1000, height=550
)

fig.show()

/var/folders/25/w29tdxyn7038m4w51t30kkgh0000gn/T/ipykernel_64477/518994707.py:8: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



# **<u> Model 1 - What separates a Passive from a Promoter? </u>**

## **<u> 1. Variable Selection </u>**

### 1.1 Correlation Analysis

- Identify variables with low or no correlation to drop


Encoding of Categorical Variables for correlation analysis

In [176]:
#Encoding of categorical variables for correlation analysis

df_pp = df_pasivos_promotores.copy()

#Simple binaries — Yes=1, No=0, Not applicable=NaN
df_pp["operacion_reciente_num"] = df_pp["operacion_reciente"].map(
    {"si": 1, "no": 0}
)  #Not applicable stays NaN

df_pp["resolucion_primer_contacto_num"] = df_pp["resolucion_primer_contacto"].map(
    {"si": 1, "no": 0}
)  #Not applicable stays NaN

#Seniority — from shortest to longest time as a customer
orden_antiguedad = {
    "menos de 6 meses": 1,
    "6-12 meses":        2,
    "1–3 años":          3,
    "3–5 años":          4,
    "más de 5 años":     5,
    "no lo sabe / prefiere no responder": None  # NaN
}
df_pp["antiguedad_num"] = df_pp["antiguedad"].map(orden_antiguedad).astype(float)

#Main channel — One-Hot
df_pp = pd.get_dummies(
    df_pp,
    columns=["canal_principal"],
    prefix="canal",
    drop_first=False,
    dtype=float
)

#Institution — One-Hot
df_pp = pd.get_dummies(
    df_pp,
    columns=["institucion"],
    prefix="inst",
    drop_first=False,
    dtype=float
)

In [177]:
#Correlation of all variables against es_promotor
complete_corr = (
    df_pp.select_dtypes(include="number")
    .corr(method="spearman")["es_promotor"]
    .drop(["es_promotor", "nps", "mes", "trimestre"])
    .sort_values(ascending=False)
)

print(complete_corr)

csat                                0.594717
ces                                 0.509463
cal_atencion_humana_num             0.425288
cal_rapidez_num                     0.398435
cal_transparencia_num               0.385016
cal_estabilidad_digital_num         0.381007
cal_facilidad_digital_num           0.379900
resolucion_primer_contacto_num      0.305929
cal_confianza_num                   0.276363
antiguedad_num                      0.161867
canal_sitio web (banca en línea)    0.103404
inst_banco azteca                   0.097754
inst_nu                             0.084171
inst_inbursa                        0.084171
inst_scotiabank                     0.061215
inst_monex                          0.059337
canal_teléfono (call center)        0.059337
inst_hsbc                           0.052351
inst_citibanamex                    0.037238
canal_app móvil                     0.007633
canal_sucursal física              -0.003515
inst_bbva                          -0.026893
inst_merca

Con base en los resultados anteriores, se dropean las variables con muy baja correlación (< 0.35) o con correlación negativa

In [178]:
#Filter to remove non-significant variables from the model
relevant_cols = []
non_relevant_cols = []

for var, corr in complete_corr.items():
    if corr >= 0.35:
        relevant_cols.append(var)
    else:
        non_relevant_cols.append(var)

print(f"Relevant variables ({len(relevant_cols)}): {relevant_cols}")
print(f"\nNon-relevant variables ({len(non_relevant_cols)}): {non_relevant_cols}")

df_pp = df_pp.drop(columns=non_relevant_cols)

print(f"\nShape after drop: {df_pp.shape}")

Relevant variables (7): ['csat', 'ces', 'cal_atencion_humana_num', 'cal_rapidez_num', 'cal_transparencia_num', 'cal_estabilidad_digital_num', 'cal_facilidad_digital_num']

Non-relevant variables (24): ['resolucion_primer_contacto_num', 'cal_confianza_num', 'antiguedad_num', 'canal_sitio web (banca en línea)', 'inst_banco azteca', 'inst_nu', 'inst_inbursa', 'inst_scotiabank', 'inst_monex', 'canal_teléfono (call center)', 'inst_hsbc', 'inst_citibanamex', 'canal_app móvil', 'canal_sucursal física', 'inst_bbva', 'inst_mercado pago', 'canal_cajero automático', 'inst_banorte', 'inst_santander', 'canal_asesor/ejecutivo', 'inst_banco nu', 'inst_invex', 'inst_invex banco', 'canal_no aplica / no lo recuerda']

Shape after drop: (166, 33)


In [179]:
#Calculate full correlation matrix (no mask)
corr_matrix = (
    df_pp.select_dtypes(include="number")
    .drop(columns=["nps", "es_promotor", "mes", "trimestre"], errors="ignore")
    .corr(method="spearman")
    .round(2)
)

fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns.tolist(),
    y=corr_matrix.columns.tolist(),
    colorscale="RdBu_r",
    zmid=0,
    zmin=-1, zmax=1,
    text=np.round(corr_matrix.values, 2),
    texttemplate="%{text}",
    textfont={"size": 9},
    hoverongaps=False
))

fig.update_layout(
    title="Correlation Matrix (Spearman)",
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(color="black"),
    width=900, height=700,
    xaxis=dict(tickangle=45, tickfont=dict(size=10)),
    yaxis=dict(tickfont=dict(size=10), autorange="reversed")
)

fig.show()

### 1.2 Multicollinearity

- Drop variables with high VIF

Reference:

Targeting Multicollinearity with Python — Towards Data Science

https://towardsdatascience.com/targeting-multicollinearity-with-python-3bd3b4088d0b/



In [180]:
#Select only relevant numerical variables
#nps (target), mes and trimestre (not features) are dropped
#NaN values are dropped because VIF does not accept missing values
cols_vif = (
     df_pp.select_dtypes(include="number")
    .drop(columns=["nps", "mes", "trimestre"])
    .dropna()
)

#Add constant required by statsmodels for the calculation
X_vif = add_constant(cols_vif)

#Calculate VIF for each variable
#VIF > 10 indicates high multicollinearity → candidate to drop
vif_df = pd.DataFrame({
    "Variable": cols_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i + 1)
            for i in range(len(cols_vif.columns))]
}).sort_values("VIF", ascending=False).reset_index(drop=True)

print(vif_df)

                      Variable       VIF
0        cal_transparencia_num  2.840926
1              cal_rapidez_num  2.751820
2  cal_estabilidad_digital_num  2.662901
3      cal_atencion_humana_num  2.369744
4    cal_facilidad_digital_num  2.348856
5                         csat  2.058976
6                          ces  1.761573
7                  es_promotor  1.678238


### 1.3 Bias and Weight Analysis

- If there is significant imbalance, apply class weights or balancing techniques

In [181]:
#Count and percentage by age group
edad_counts = (
    df_pasivos_promotores["edad"].value_counts()
    .reset_index()
    .rename(columns={"edad": "age_group", "count": "count"})
)
edad_counts["pct"] = (edad_counts["count"] / len(df_est) * 100).round(1)
edad_counts["label"] = edad_counts.apply(
    lambda r: f"{r['count']}<br>({r['pct']}%)", axis=1
)

age_label_map = {"65 o más": "65+"}
edad_counts["age_group"] = edad_counts["age_group"].replace(age_label_map)

fig = px.bar(
    edad_counts,
    x="age_group",
    y="count",
    text="label",
    title="Age distribution",
    labels={"age_group": "", "count": "Number of responses"},
    color_discrete_sequence=["#2a9d8f"]
)

fig.update_traces(textposition="outside")
fig.update_layout(
    plot_bgcolor="#F5F5F0",
    paper_bgcolor="#F5F5F0",
    font=dict(color="#333333", size=12),
    title_font=dict(size=16, color="#1a1a2e", family="Arial Black"),
    xaxis=dict(showgrid=False, tickfont=dict(color="#333333", size=12), linecolor="#cccccc"),
    yaxis=dict(gridcolor="#e0e0e0", tickfont=dict(color="#333333", size=12)),
    showlegend=False,
    width=900, height=550
)

fig.show()

### 1.4 Final Variable Selection


In [182]:
features = [
    "cal_atencion_humana_num",
    "cal_rapidez_num",
    "cal_transparencia_num",
    "cal_estabilidad_digital_num",
    "cal_facilidad_digital_num"
]

### 1.5 Correlation with Final Features

In [183]:
#Correlation of selected features against es_promotor
#Only on the Passives and Promoters segment
correlacion_promotor = (
    df_pp[features + ["es_promotor"]]
    .corr(method="spearman")["es_promotor"]
    .drop("es_promotor")
    .sort_values(ascending=False)
)

print(correlacion_promotor)

cal_atencion_humana_num        0.425288
cal_rapidez_num                0.398435
cal_transparencia_num          0.385016
cal_estabilidad_digital_num    0.381007
cal_facilidad_digital_num      0.379900
Name: es_promotor, dtype: float64


**Low correlation variables:** The correlation analysis was performed on the Passives and Promoters segment using Spearman's method. A minimum threshold of 0.35 was set to consider a variable relevant. 0.35 represents a balance between real signal and model simplicity — it includes only variables with moderate-to-strong correlation and excludes those that could be statistical noise given the sample size.

**Multicollinearity:** The VIF analysis confirms there is no problematic multicollinearity.

**CES and CSAT:** These are excluded from the model despite being the strongest NPS predictors. Historically, these metrics are designed to correlate with NPS — including them would not generate a new finding but would simply confirm something already known.

**Age bias:** The dataset is skewed toward the 18-24 age group, which accounts for 36.3% of responses. To mitigate its impact, the following measures will be taken:

- `class_weight="balanced"` in the model to correct the class imbalance (Passive/Promoter)
- Stratification in the split to maintain similar proportions of each group across the three sets

Given the small dataset size (193 rows), it will be evaluated whether these measures have a positive impact on model performance or introduce distortions. The final decision will be made based on the results.

**Selected features:** Based on the previous analyses, the features entering the model are the perceived quality variables — cal_rapidez_num, cal_transparencia_num, cal_facilidad_digital_num, cal_estabilidad_digital_num. A total of 5 features, all with correlation >= 0.35 with es_promotor and no problematic multicollinearity.

## **<u> 2. Data Preparation </u>**

### 2.1 Handle Empty Columns or NaN Values

Two possible paths to evaluate:
- Impute empty columns with the median
- Drop the rows and lose 19% of the data

In [184]:
#Check NaN values in features before dropping
print("NaN per variable:")
print(df_pp[features].isna().sum())
print(f"\nTotal rows before: {len(df_pp)}")

#Drop rows with NaN in features
df_pp = df_pp.dropna(subset=features)

print(f"Total rows after: {len(df_pp)}")
print(f"Rows lost: {len(df_pasivos_promotores) - len(df_pp)}")

NaN per variable:
cal_atencion_humana_num        25
cal_rapidez_num                19
cal_transparencia_num          20
cal_estabilidad_digital_num    19
cal_facilidad_digital_num      20
dtype: int64

Total rows before: 166
Total rows after: 134
Rows lost: 32


### 2.1.1 Distribution Review

In [185]:
#Distribution of variables with NaN — to evaluate whether median imputation is valid
variables_nan = [
    "cal_atencion_humana_num",
    "cal_rapidez_num",
    "cal_transparencia_num",
    "cal_estabilidad_digital_num",
    "cal_facilidad_digital_num"
]

fig = make_subplots(
    rows=1, cols=5,
    subplot_titles=variables_nan
)

for i, var in enumerate(variables_nan, start=1):
    fig.add_trace(
        go.Histogram(
            x=df_pp[var].dropna(),
            marker_color="#2a9d8f",
            name=var,
            showlegend=False,
            xbins=dict(start=0.5, end=5.5, size=1)
        ),
        row=1, col=i
    )

fig.update_layout(
    title=dict(text="Distribution of variables with NaN", x=0.5,
               font=dict(size=16, color="#1a1a2e", family="Arial Black")),
    plot_bgcolor="#F5F5F0",
    paper_bgcolor="#F5F5F0",
    font=dict(color="#333333", size=11),
    height=400,
    width=1400
)

fig.update_xaxes(tickvals=[1, 2, 3, 4, 5],
                 ticktext=["Very bad", "Bad", "Average", "Good", "Very good"],
                 tickangle=45, tickfont=dict(size=9))
fig.update_yaxes(gridcolor="#e0e0e0")

fig.show()

Decision based on distributions:

The distributions show a marked skew toward high values — most customers rated "Good" or "Very good" across all variables. Given this skew, imputing with mean, median, or mode would introduce artificially high values that do not reflect the reality of customers who did not respond.

For this reason, rows with NaN are dropped, working only with complete responses.

In [186]:
#Check how many NaN values each row has
print(df_pp[features].isna().sum(axis=1).value_counts().sort_index())

0    134
Name: count, dtype: int64


### 2.1 Data Split
- Train (70%) — model training
- Test (30%) — final evaluation



Why 70-30?

Given the small dataset size (134 rows), a 70/30 split was used for base training. Increasing the test set yields more representative metrics of real performance, though it implies a tradeoff — the model trains on less data and may lose some generalization capacity. With a sample of this size, that sacrifice is acceptable for a fair comparison between models.


In [187]:
X = df_pp[features]
y = df_pp["es_promotor"]

#Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y       #Stratification by class
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")


Train: (93, 5), Test: (41, 5)


In [188]:
print(df_pp["es_promotor"].value_counts())
print(df_pp["es_promotor"].value_counts(normalize=True).mul(100).round(1))

es_promotor
1    90
0    44
Name: count, dtype: int64
es_promotor
1    67.2
0    32.8
Name: proportion, dtype: float64


## **<u>3. Training and Tuning </u>**


### 3.1 Model Selection and Tuning

3 classification models will be evaluated. For each one, a 5-fold cross-validated GridSearchCV will be run to find the optimal hyperparameters. The models are:

    - Logistic Regression
    - SVM (Support Vector Machine)
    - Decision Tree

The model with the highest Recall on the test set will be selected as the final model. Ensemble models will not be used due to the limited data size.


In [189]:
#Model definitions (with class weights to handle bias)

#Logistic Regression
model_lr = LogisticRegression(
    class_weight="balanced",
    random_state=42,
    max_iter=1000
)

#SVM
model_svm = SVC(
    class_weight="balanced",
    random_state=42,
    probability=True
)

#Decision Tree
model_dt = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=42,
    max_depth=4
)

### 3.2 Results and Model Selection

A 5-fold cross-validated GridSearchCV was run for each model optimizing for Recall — the priority metric given that the cost of missing a potential Promoter is higher than a false positive.

The comparison table shows Accuracy, Precision, Recall, and F1 on the test set for each model's best configuration. The model with the highest test Recall is selected for final evaluation.

In [190]:
models = {
    "Logistic Regression": model_lr,
    "SVM":                 model_svm,
    "Decision Tree":       model_dt
}

#Hyperparameter grids for each model
param_grids = {
    "Logistic Regression": {"C": [0.001, 0.01, 0.1, 1, 10, 100]},
    "SVM":                 {"C": [0.01, 0.1, 1, 10, 100], "kernel": ["linear", "rbf"]},
    "Decision Tree":       {"max_depth": [2, 3, 4, 5, None], "min_samples_split": [2, 5, 10]}
}

#Run GridSearchCV for each model — CV Recall on train set only, test set stays unseen
results = {}

for name, model in models.items():
    grid = GridSearchCV(
        estimator=model,
        param_grid=param_grids[name],
        cv=5,
        scoring="recall"
    )
    grid.fit(X_train, y_train)

    results[name] = {
        "best_params":    grid.best_params_,
        "cv_recall":      grid.best_score_,
        "best_estimator": grid.best_estimator_
    }
    print(f"{name} — Best params: {grid.best_params_} | CV Recall: {grid.best_score_:.3f}")

#Comparison table — CV Recall only
results_df = pd.DataFrame(results).T.drop(columns=["best_params", "best_estimator"])
print("\nModel Comparison (CV Recall):")
print(results_df.astype(float).round(3))



Logistic Regression — Best params: {'C': 0.001} | CV Recall: 0.806
SVM — Best params: {'C': 0.01, 'kernel': 'linear'} | CV Recall: 0.872
Decision Tree — Best params: {'max_depth': None, 'min_samples_split': 10} | CV Recall: 0.710

Model Comparison (CV Recall):
                     cv_recall
Logistic Regression      0.806
SVM                      0.872
Decision Tree            0.710


### 3.3 Model Selection — Logistic Regression

Model selection was based solely on CV Recall from GridSearchCV, keeping the test set completely unseen. With only 134 rows, a separate validation split was not feasible — 5-fold cross-validation served as the validation strategy.

| Model | CV Recall |
|---|---|
| Logistic Regression | 0.806 |
| SVM | 0.872 |
| Decision Tree | 0.710 |

Despite SVM achieving a slightly higher CV Recall, Logistic Regression is selected for the following reasons:

- **Interpretability:** Coefficients directly quantify each feature's contribution to the probability of being a Promoter — central to the hypothesis and business value of the model.
- **Simplicity and overfitting risk:** With only 5 features and 134 rows, a linear decision boundary is appropriate. SVM's higher CV Recall is likely attributable to fold variance rather than a real generalization advantage — each fold contains ~27 rows, which is too small to produce stable estimates. A more complex model carries a real overfitting risk at this sample size.
- **CV gap within variance range:** The difference in CV Recall between both models is within a range attributable to fold variance given the small sample size, making LR the safer and more justified choice.

Decision Tree is discarded due to significantly lower CV Recall (0.710).

### 3.4 Hyperparameter Tuning — Logistic Regression

With Logistic Regression selected, a focused tuning step is run to find the optimal `C` value using 5-fold cross-validation on the train set only — test set remains unseen.

- **C:** Controls regularization strength. Lower values = stronger regularization, reducing overfitting risk at small sample sizes.
- **Solver:** `liblinear` is used instead of the default `lbfgs` — it is specifically designed for small datasets, converges reliably without iteration tuning, and handles L2 regularization efficiently.

In [191]:
#Focused tuning for Logistic Regression — train set only
param_grid_lr = {
    "C":        [0.01, 0.1, 1, 10, 100, 1000],
    "max_iter": [50, 100, 200, 500, 1000]
}

grid_lr = GridSearchCV(
    estimator=LogisticRegression(
        class_weight="balanced",
        random_state=42
    ),
    param_grid=param_grid_lr,
    cv=5,
    scoring="recall"
)

grid_lr.fit(X_train, y_train)

print(f"Best params:    {grid_lr.best_params_}")
print(f"Best CV Recall: {grid_lr.best_score_:.3f}")

model_lr_tuned = grid_lr.best_estimator_

Best params:    {'C': 0.01, 'max_iter': 50}
Best CV Recall: 0.806


In [192]:
y_pred_train = model_lr_tuned.predict(X_train)
print(f"Unique predictions: {np.unique(y_pred_train)}")
print(f"Predicted Promoter %: {y_pred_train.mean():.2%}")

Unique predictions: [0 1]
Predicted Promoter %: 65.59%




The optimal configuration found was `C=0.01` with `max_iter=50`, achieving a CV Recall of **0.806** on the train set. The low C value confirms that strong regularization is appropriate given the small dataset size. The prediction distribution on the train set (65.6% Promoter) aligns with the actual class distribution, confirming the model is not collapsing to a trivial solution.

## **<u> 4. Final Evaluation on Test Set </u>**

In [193]:
#Final evaluation of tuned Logistic Regression on test set
y_pred = model_lr_tuned.predict(X_test)
y_prob = model_lr_tuned.predict_proba(X_test)[:, 1]

### 4.1 Métricas del Modelo 


In [194]:
print("=" * 40)
print("FINAL EVALUATION — Logistic Regression")
print("=" * 40)
print(classification_report(y_test, y_pred,
                            target_names=["Passive", "Promoter"]))
print(f"AUC-ROC: {round(roc_auc_score(y_test, y_prob), 3)}")

FINAL EVALUATION — Logistic Regression
              precision    recall  f1-score   support

     Passive       0.60      0.69      0.64        13
    Promoter       0.85      0.79      0.81        28

    accuracy                           0.76        41
   macro avg       0.72      0.74      0.73        41
weighted avg       0.77      0.76      0.76        41

AUC-ROC: 0.808


### Metric Interpretation

The model meets its primary objective with a **Recall of 0.79 for Promoters** — it correctly identifies 79% of actual Promoters, which is the most important metric for this project. A Precision of 0.85 means that when it predicts a Promoter, it is correct 85% of the time. The AUC-ROC of 0.808 confirms the model has strong discriminative power between the two classes.

The main weakness is in Passives — Recall of 0.69 — meaning the model correctly identifies 69% of actual Passives. This is expected given the class imbalance (13 Passives vs 28 Promoters in the test set) and the limited number of examples available for the model to learn Passive patterns.


### 4.2 ROC Curve

In [195]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=fpr, y=tpr,
    mode="lines",
    name=f"ROC (AUC = {auc_score:.3f})",
    line=dict(color="#2a9d8f", width=2)
))

fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode="lines",
    name="Random classifier",
    line=dict(color="#e63946", width=1.5, dash="dash")
))

fig.update_layout(
    title="ROC Curve — Logistic Regression",
    plot_bgcolor="#F5F5F0",
    paper_bgcolor="#F5F5F0",
    font=dict(color="#333333", size=12),
    title_font=dict(size=16, color="#1a1a2e", family="Arial Black"),
    xaxis=dict(title="False Positive Rate", gridcolor="#e0e0e0",
               tickfont=dict(color="#333333")),
    yaxis=dict(title="True Positive Rate", gridcolor="#e0e0e0",
               tickfont=dict(color="#333333")),
    legend=dict(font=dict(color="#333333")),
    width=700, height=550
)

fig.show()

### ROC Curve Interpretation

With an **AUC-ROC of 0.808**, the model correctly distinguishes between Passives and Promoters in 80.8% of cases — significantly better than a random classifier (0.50).

The stepped shape is characteristic of small datasets — with only 41 rows in the test set, each prediction generates a visible jump rather than a smooth curve. This does not indicate a problem with the model but rather a limitation of the sample size.

### 4.3 Confusion Matrix

In [196]:
cm = confusion_matrix(y_test, y_pred)
cm_flipped = cm[::-1, ::-1]

fig = ff.create_annotated_heatmap(
    z=cm_flipped,
    x=["Pred Promoter", "Pred Passive"],
    y=["Real Promoter", "Real Passive"],
    colorscale=[[0, "#F5F5F0"], [1, "#2a9d8f"]],
    showscale=False,
    annotation_text=cm_flipped
)

fig.update_layout(
    title=dict(text="Confusion Matrix", x=0.5,
               font=dict(size=16, color="#1a1a2e", family="Arial Black")),
    plot_bgcolor="#F5F5F0",
    paper_bgcolor="#F5F5F0",
    font=dict(color="#333333", size=12),
    width=500, height=400
)

fig.show()

### Confusion Matrix Interpretation

The model correctly identifies **22 of 28 actual Promoters** (Recall 0.79) and misses 6. For Passives it correctly identifies 9 of 13 — consistent with their lower representation in the data. The 4 Passives misclassified as Promoters are the most frequent error and reflect the difficulty of distinguishing this class with so few examples.

## **<u> 5. Results: What separates a Passive from a Promoter? </u>**

### 5.1 Feature Importance
- Plot of most important variables for Passive → Promoter conversion
- Ranking of actionable levers for the bank

In [197]:
#Treemap  
coeficientes_tree = pd.DataFrame({
    "variable": features,
    "importancia": model_lr_tuned.coef_[0]
})

#Normalize coefficients to relative importance as percentage
#Absolute value of each coefficient divided by total sum
#Each variable is expressed as its percentage contribution to the model
coeficientes_tree["importancia"] = (
    coeficientes_tree["importancia"].abs() / 
    coeficientes_tree["importancia"].abs().sum() * 100
).round(1)

coeficientes_tree["variable"] = (
    coeficientes_tree["variable"]
    .str.replace("_num", "")
    .str.replace("cal_", "")
    .str.replace("_", " ")
    .str.title()
    
)

name_map = {
    "Facilidad Digital": "Digital Ease",
    "Estabilidad Digital": "Digital Stability",
    "Transparencia": "Transparency",
    "Rapidez": "Speed",
    "Atencion Humana": "Human Support"
}

coeficientes_tree["variable"] = coeficientes_tree["variable"].map(name_map)

coeficientes_tree["label"] = coeficientes_tree.apply(
    lambda r: f"{r['variable']}<br>{r['importancia']:.1f}%", axis=1
)


fig = px.treemap(
    coeficientes_tree,
    path=["variable"],
    values="importancia",
    color="importancia",
    color_continuous_scale=[
        [0.0, "#f4a261"],
        [0.5, "#e63946"],
        [1.0, "#7b0000"]
    ],
    color_continuous_midpoint=coeficientes_tree["importancia"].mean(),
    title="Feature Importance — Passive → Promoter Conversion Drivers",
    custom_data=["importancia"]
)

fig.update_traces(
    texttemplate="<b>%{label}</b><br>%{customdata[0]:.1f}%",
    textfont=dict(size=14, color="white")
)

fig.update_layout(
    plot_bgcolor="#F5F5F0",
    paper_bgcolor="#F5F5F0",
    font=dict(color="#333333", size=12),
    title_font=dict(size=16, color="#1a1a2e", family="Arial Black"),
    coloraxis_showscale=False,
    width=900, height=550
)

fig.show()

### Feature Importance Interpretation

The five variables have balanced importance, indicating that the conversion from Passive to Promoter depends on an integral experience rather than a single lever.

**Digital Ease (24.9%)** and **Digital Stability (21.4%)** are the factors that most differentiate the two groups — a Promoter perceives they can resolve their needs easily through digital channels. **Transparency (18.6%)** confirms the project's initial hypothesis.

Together, the digital experience — Digital Ease and Digital Stability — accounts for 46.3% of total importance and represents the greatest conversion opportunity for the bank.

### 5.2 Comparative Profile
- Typical Passive vs typical Promoter profile across key variables

In [198]:
perfil = (
    df_pp.groupby("es_promotor")[features]
    .mean()
    .round(2)
)

gap = pd.DataFrame({
    "variable": features,
    "passive":  perfil.loc[0].values,
    "promoter": perfil.loc[1].values
})

gap["difference"] = (gap["promoter"] - gap["passive"]).round(2)
gap = gap.sort_values("difference", ascending=True)

gap["variable"] = (
    gap["variable"]
    .str.replace("_num", "")
    .str.replace("cal_", "")
    .str.replace("_", " ")
    .str.title()
)

name_map = {
    "Facilidad Digital": "Digital Ease",
    "Estabilidad Digital": "Digital Stability",
    "Transparencia": "Transparency",
    "Rapidez": "Speed",
    "Atencion Humana": "Human Support"
}
gap["variable"] = gap["variable"].map(name_map)

fig = px.bar(
    gap,
    x="difference",
    y="variable",
    orientation="h",
    text=gap["difference"].apply(lambda x: f"+{x:.2f}"),
    title="Experience Gap — How far is a Passive from becoming a Promoter?",
    labels={"difference": "Difference in average score", "variable": ""},
    color_discrete_sequence=["#2a9d8f"]
)

fig.update_traces(textposition="outside", cliponaxis=False)
fig.update_layout(
    plot_bgcolor="#F5F5F0",
    paper_bgcolor="#F5F5F0",
    font=dict(color="#333333", size=12),
    title_font=dict(size=16, color="#1a1a2e", family="Arial Black"),
    xaxis=dict(tickfont=dict(color="#333333", size=11), linecolor="#cccccc"),
    yaxis=dict(tickfont=dict(color="#333333", size=11), gridcolor="#e0e0e0"),
    showlegend=False,
    width=900, height=500
)

fig.show()

### 5.3 Why is this model valuable for the bank?

The value of the model is not just in predicting — it is in what it reveals about the customer experience and the levers to improve it.

**Retain Promoters.** The bank can identify which customers have a Promoter profile and ensure the experience that got them there is maintained before they drop to Passive.

**Convert Passives.** For each Passive customer the model delivers a conversion probability and signals which variables are underperforming — giving the bank a specific direction to act on.

**Prioritize actions.** Not all Passives are equal. The model allows ranking them by conversion probability and focusing resources where the impact on NPS is greatest.

The model turns NPS from an outcome indicator into an active customer management tool.

# **<u> 6. Conclusions </u>**

## Main Finding

The analysis identified that **Digital Ease, Human Support, and Transparency** are the factors that most differentiate a Passive customer from a Promoter. A Passive does not have a radically different profile from a Promoter — they simply rate these variables slightly lower. This means that small improvements in the experience may be enough to convert them.

## Model Performance

The model correctly identifies **79% of actual Promoters** with an AUC-ROC of 0.808 — a solid result considering the small dataset size. The main limitation is in the detection of Passives, directly related to their low representation in the data.

## Why implement the model?

With the model the bank can:

- **Identify Promoters** and ensure the experience that got them there is maintained before they drop to Passives.
- **Prioritize Passives** by ranking them by conversion probability and focusing resources where the impact on NPS is greatest.
- **Act on data** — not intuition. Every decision about where to improve the experience will be backed by the real weight of each variable.

## Limitations

The main factor limiting the model's scope is the **dataset size (193 rows)**. With a small sample the model has fewer examples to learn patterns from, which is reflected especially in the low detection of Passives — the class with the least representation.

Additionally, the dataset presents two important biases:

- **Class bias:** 67% of customers in the analyzed segment are Promoters, which favors the detection of that class over Passives.
- **Age bias:** 36.3% of responses come from the 18-24 age group, so the results mainly reflect the experience of young customers and may not generalize to other segments.

With a larger, balanced, and more representative sample across all age segments, the model could identify more robust patterns, improve Passive detection, and deliver even more precise and reliable levers for the bank.

## Recommendation

To maximize the value of the model, it is recommended to expand the sample of Passive customers in future surveys. With more data the model would significantly improve its detection capacity and the identified levers would be even more precise and reliable.

## **<u> 7. Extra Validation — SMOTE </u>**

### What is SMOTE?

SMOTE (Synthetic Minority Over-sampling Technique) is an algorithm that generates artificial data for the minority class. Instead of duplicating existing records — which would cause overfitting — SMOTE creates new synthetic points by interpolating between the k nearest neighbors of the minority class in the feature space.

The process is:
1. Takes a point from the minority class (Passives)
2. Finds its k nearest neighbors
3. Creates a new point at a random position between that point and one of its neighbors
4. Repeats until the classes are balanced

**Important:** SMOTE is only applied to the training set — never to the test set. Applying it to the full dataset would cause data leakage and the metrics would not reflect the model's real performance.

Reference: Create Artificial Data With SMOTE — Towards Data Science
https://medium.com/data-science/create-artificial-data-with-smote-2a31ee855904

### 7.1 Aplicar SMOTE al train set y verificar balance de clases

In [199]:
#Apply SMOTE only to the train set
#Test set remains untouched
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

#Verify class balance
print("Distribution BEFORE SMOTE:")
print(pd.Series(y_train).value_counts())
print(f"Total: {len(y_train)}")

print("\nDistribution AFTER SMOTE:")
print(pd.Series(y_train_smote).value_counts())
print(f"Total: {len(y_train_smote)}")

Distribution BEFORE SMOTE:
es_promotor
1    62
0    31
Name: count, dtype: int64
Total: 93

Distribution AFTER SMOTE:
es_promotor
0    62
1    62
Name: count, dtype: int64
Total: 124


### 7.2 Train model with balanced data

In [200]:
#class_weight="balanced" is removed — classes are already balanced by SMOTE
model_smote = LogisticRegression(
    C=grid_lr.best_params_["C"],
    max_iter=grid_lr.best_params_["max_iter"],
    random_state=99
)

model_smote.fit(X_train_smote, y_train_smote)
print("Model trained with SMOTE-balanced data")

Model trained with SMOTE-balanced data


### 7.3 Evaluation and Metric Comparison — with and without SMOTE

In [201]:
#Evaluation and metric comparison
y_pred_smote = model_smote.predict(X_test)
y_prob_smote = model_smote.predict_proba(X_test)[:, 1]

print("=" * 40)
print("ORIGINAL MODEL")
print("=" * 40)
print(classification_report(y_test, y_pred,
                            target_names=["Passive", "Promoter"]))
print(f"AUC-ROC: {round(roc_auc_score(y_test, y_prob), 3)}")

print("\n" + "=" * 40)
print("SMOTE MODEL")
print("=" * 40)
print(classification_report(y_test, y_pred_smote,
                            target_names=["Passive", "Promoter"]))
print(f"AUC-ROC: {round(roc_auc_score(y_test, y_prob_smote), 3)}")

ORIGINAL MODEL
              precision    recall  f1-score   support

     Passive       0.60      0.69      0.64        13
    Promoter       0.85      0.79      0.81        28

    accuracy                           0.76        41
   macro avg       0.72      0.74      0.73        41
weighted avg       0.77      0.76      0.76        41

AUC-ROC: 0.808

SMOTE MODEL
              precision    recall  f1-score   support

     Passive       0.60      0.69      0.64        13
    Promoter       0.85      0.79      0.81        28

    accuracy                           0.76        41
   macro avg       0.72      0.74      0.73        41
weighted avg       0.77      0.76      0.76        41

AUC-ROC: 0.809


### 7.4 Conclusion — SMOTE

SMOTE did not improve Passive Recall because the limitation is data-related, not technical. With only 44 real Passives the model does not have enough examples to learn to distinguish them precisely — no artificial balancing technique can solve that.

The real solution is to collect more responses from Passive customers.

# **<u> 8. Extra — Churn Risk Metric </u>**

An operational metric is proposed that the bank can use immediately to identify customers at risk of churning using data it already has.

### Why is it relevant?

The main model focuses on converting Passives into Promoters. This metric complements that focus by identifying Detractors at risk of churning — the segment the model does not address directly.

Together they cover the full spectrum of active customer management:

- At-risk customer → act before losing them
- Passive customer → convert them into a Promoter
- Promoter customer → maintain their experience

### 8.1 Metric Construction

A customer is considered at risk if they meet at least 2 of these 3 conditions:

| Condition | Threshold | Justification |
|-----------|-----------|---------------|
| NPS | ≤ 6 (Detractor) | Low probability of recommending |
| CSAT | ≤ 3 | Low overall satisfaction |
| CES | ≥ 3 | High friction in the experience |

In [202]:
#Churn proxy construction
#At risk if at least 2 of 3 conditions are met
df_churn = df_est.copy()

df_churn["churn_score"] = (
    (df_churn["nps"] <= 6).astype(int) +
    (df_churn["csat"] <= 3).astype(int) +
    (df_churn["ces"] >= 3).astype(int)
)

df_churn["churn_riesgo"] = (df_churn["churn_score"] >= 2).astype(int)

print("Distribution of at-risk customers:")
print(df_churn["churn_riesgo"].value_counts())
print(df_churn["churn_riesgo"].value_counts(normalize=True).mul(100).round(1))

Distribution of at-risk customers:
churn_riesgo
0    159
1     34
Name: count, dtype: int64
churn_riesgo
0    82.4
1    17.6
Name: proportion, dtype: float64


In [203]:
#Distribution of at-risk vs non-at-risk customers
labels = {0: "No risk", 1: "At risk"}
df_churn["churn_label"] = df_churn["churn_riesgo"].map(labels)

conteo = (
    df_churn["churn_label"].value_counts()
    .reset_index()
    .rename(columns={"churn_label": "group", "count": "count"})
)
conteo["pct"] = (conteo["count"] / len(df_churn) * 100).round(1)
conteo["label"] = conteo.apply(lambda r: f"{r['count']}<br>({r['pct']}%)", axis=1)

fig = px.bar(
    conteo,
    x="group",
    y="count",
    color="group",
    text="label",
    title="Customers at Churn Risk",
    labels={"group": "", "count": "Number of customers"},
    color_discrete_map={"No risk": "#2a9d8f", "At risk": "#e63946"}
)

fig.update_traces(textposition="outside")
fig.update_layout(
    plot_bgcolor="#F5F5F0",
    paper_bgcolor="#F5F5F0",
    font=dict(color="#333333", size=12),
    title_font=dict(size=16, color="#1a1a2e", family="Arial Black"),
    xaxis=dict(tickfont=dict(color="#333333"), linecolor="#cccccc"),
    yaxis=dict(gridcolor="#e0e0e0", tickfont=dict(color="#333333")),
    showlegend=False,
    width=700, height=500
)

fig.show()

### 8.2 Conclusion

This metric is an approximation based on dissatisfaction signals available in the survey. It is not a real churn model — its validity depends on whether the defined conditions effectively correlate with actual customer churn.

If the bank begins tracking which customers actually leave, this metric can evolve into a supervised predictive model with greater precision and confidence.